In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
cancer = load_breast_cancer()

X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name="target")

print("Dataset shape:", X.shape)
print("Target classes:", cancer.target_names)
print("\nFirst five rows:")
display(X.head())

print("\nClass distribution:")
print(y.value_counts().rename(index={0: cancer.target_names[0], 1: cancer.target_names[1]}))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

In [ ]:
linear_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(kernel="linear"))
])

rbf_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(kernel="rbf"))
])

linear_svm.fit(X_train, y_train)
rbf_svm.fit(X_train, y_train)

print("Linear SVM training completed.")
print("RBF SVM training completed.")

In [ ]:
y_pred_linear = linear_svm.predict(X_test)
y_pred_rbf = rbf_svm.predict(X_test)

print("Predictions generated for both models.")

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear SVM", "RBF SVM"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_linear),
        accuracy_score(y_test, y_pred_rbf)
    ],
    "Precision": [
        precision_score(y_test, y_pred_linear),
        precision_score(y_test, y_pred_rbf)
    ],
    "Recall": [
        recall_score(y_test, y_pred_linear),
        recall_score(y_test, y_pred_rbf)
    ],
    "F1-Score": [
        f1_score(y_test, y_pred_linear),
        f1_score(y_test, y_pred_rbf)
    ]
})

display(results.round(4))

In [ ]:
print("Linear SVM Classification Report")
print(classification_report(y_test, y_pred_linear, target_names=cancer.target_names))

print("RBF SVM Classification Report")
print(classification_report(y_test, y_pred_rbf, target_names=cancer.target_names))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_linear,
    display_labels=cancer.target_names,
    cmap="Blues",
    ax=axes[0]
)
axes[0].set_title("Linear SVM Confusion Matrix")

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_rbf,
    display_labels=cancer.target_names,
    cmap="Greens",
    ax=axes[1]
)
axes[1].set_title("RBF SVM Confusion Matrix")

plt.tight_layout()
plt.show()

In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]

results_plot = results.set_index("Model")[metrics]
results_plot.plot(kind="bar", figsize=(10, 6), ylim=(0, 1.05))

plt.title("Performance Comparison: Linear SVM vs RBF SVM")
plt.ylabel("Score")
plt.xlabel("SVM Model")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
best_model = results.loc[results["F1-Score"].idxmax(), "Model"]
best_f1 = results["F1-Score"].max()

print("Best model based on F1-Score:", best_model)
print("Best F1-Score:", round(best_f1, 4))

print("\nFinal comparison:")
display(results.sort_values(by="F1-Score", ascending=False).round(4))